In [8]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [5]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [6]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [7]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [20]:
mlflow.set_experiment("Anamoly Detection")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

for idx, element in enumerate(models):
    report = reports[idx]
    model_name = element[0]
    model = element[2]
    params = element[1]
    with mlflow.start_run(run_name=model_name):

        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, name=model_name)
        else:
            mlflow.sklearn.log_model(model, name=model_name, skops_trusted_types=["sklearn.tree._tree.Tree"])  

2026/11/15 15:55:51 INFO mlflow.tracking.fluent: Experiment with name 'Anamoly Detection' does not exist. Creating a new experiment.
2026/11/15 15:55:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/1/runs/7f74c07a08ae4456941c57ff5888e472
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/15 15:55:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/1/runs/c5198974d89c43e889517c196079c49d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/15 15:55:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/2c5b639822aa45e5b8580cb48edf7fe2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/15 15:55:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/1/runs/7575235b83244ce5bbcb21bc8a8f36a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


# Register the model

In [31]:
run_id = input()
logged_info = mlflow.search_logged_models(filter_string = f"source_run_id = '{run_id}'", output_format='list')

 7575235b83244ce5bbcb21bc8a8f36a6


In [37]:
for model in logged_info:
    print(model.name, model.model_id)

XGBClassifier With SMOTE m-b346dbb092e94c28ba54d1e5a6916ce3


In [46]:
model_info = logged_info[0]
model_id = model_info.model_id
model_name = "XGB-Smote"
model_uri = f"models:/{model_id}"
result = mlflow.register_model(model_uri, model_name)

Successfully registered model 'XGB-Smote'.
2026/11/15 16:34:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


# Load the model

In [73]:
experiment = mlflow.get_experiment_by_name('Anamoly Detection')

logged_info = mlflow.search_logged_models(experiment_ids=[experiment.experiment_id],output_format='list', filter_string=f"name = 'XGBClassifier With SMOTE'")[0]

model_uri = f"models:/{logged_info.model_id}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [76]:
client = mlflow.MlflowClient()
reg_model_name = "XGB-Smote"
prod_model = 'anamoly-detection-prod'
version = '1'
src_uri = f"models:/{reg_model_name}/{version}"
client.copy_model_version(src_model_uri=src_uri, dst_name=prod_model)

Successfully registered model 'anamoly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anamoly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1794741854941, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1794741854941, metrics=None, model_id=None, name='anamoly-detection-prod', params=None, run_id='7575235b83244ce5bbcb21bc8a8f36a6', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>